In [13]:
from langchain_openai.chat_models import ChatOpenAI
from pydantic import BaseModel
from langchain_core.documents import Document
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
import os


In [14]:
from load_key.load_key import Settings

Settings.validate()

In [15]:
text = """Artificial intelligence is transforming technology and shaping the future.
Machine learning algorithms are becoming more sophisticated every day.
Deep learning models can now process vast amounts of data efficiently.
Neural networks are inspired by the human brain's structure.
The best pasta recipes include fresh ingredients and proper cooking techniques.
Italian cuisine emphasizes quality olive oil and regional cheeses.
Authentic carbonara uses guanciale, eggs, pecorino romano, and black pepper.
Cooking pasta al dente ensures the best texture and flavor.
Climate change is affecting ecosystems worldwide.
Rising temperatures are causing glaciers to melt at unprecedented rates.
Scientists warn that immediate action is needed to reduce carbon emissions.
Renewable energy sources offer hope for a sustainable future."""

In [16]:
# pydantic class for structured output

class Chunk(BaseModel):

    chunk_text: str
    summary: str


class Chunker(BaseModel):

    chunks: list[Chunk]

In [34]:
# define model
print(bool(Settings.GEMINI_API_KEY))
key = Settings.GEMINI_API_KEY.strip()
model = ChatOpenAI(api_key=key, model="gemini-3.6-flash", base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

llm_chunker = model.with_structured_output(schema=Chunker)


True
Hi! How can I help you today?


In [26]:
# prompt for chunking

prompt = ChatPromptTemplate(messages=[
    ("system",
     """You are an expert Text Chunker that splits the given text and outputs them as a
     list of strings. You understand the natural topic boundaries of text and
     also do not change the existing text. You just split the text where ever applicable.
     Once you create the chunk, you also generate a 1-2 line summary of the chunk also"""),
    ("human",
     "Split the given text into chunks\nText: {text}")
], input_variables=["text"])

In [36]:
# chunking through llm

model_chain = (prompt | llm_chunker).with_retry(
    stop_after_attempt=5,
    wait_exponential_jitter=True,
)
response = model_chain.invoke({"text": text})

In [37]:
response

Chunker(chunks=[Chunk(chunk_text="Artificial intelligence is transforming technology and shaping the future. Machine learning algorithms are becoming more sophisticated every day. Deep learning models can now process vast amounts of data efficiently. Neural networks are inspired by the human brain's structure.", summary='An overview of how artificial intelligence, machine learning, deep learning, and neural networks are advancing and transforming technology.'), Chunk(chunk_text='The best pasta recipes include fresh ingredients and proper cooking techniques. Italian cuisine emphasizes quality olive oil and regional cheeses. Authentic carbonara uses guanciale, eggs, pecorino romano, and black pepper. Cooking pasta al dente ensures the best texture and flavor.', summary='Key elements of Italian cuisine and essential tips for cooking authentic, high-quality pasta dishes.'), Chunk(chunk_text='Climate change is affecting ecosystems worldwide. Rising temperatures are causing glaciers to melt 

In [38]:
response.chunks

[Chunk(chunk_text="Artificial intelligence is transforming technology and shaping the future. Machine learning algorithms are becoming more sophisticated every day. Deep learning models can now process vast amounts of data efficiently. Neural networks are inspired by the human brain's structure.", summary='An overview of how artificial intelligence, machine learning, deep learning, and neural networks are advancing and transforming technology.'),
 Chunk(chunk_text='The best pasta recipes include fresh ingredients and proper cooking techniques. Italian cuisine emphasizes quality olive oil and regional cheeses. Authentic carbonara uses guanciale, eggs, pecorino romano, and black pepper. Cooking pasta al dente ensures the best texture and flavor.', summary='Key elements of Italian cuisine and essential tips for cooking authentic, high-quality pasta dishes.'),
 Chunk(chunk_text='Climate change is affecting ecosystems worldwide. Rising temperatures are causing glaciers to melt at unpreceden

In [47]:
len(response.chunks)

3

In [40]:
chunks = response.chunks

In [46]:
from termcolor import COLORS, colored
from random import choice

In [42]:
def display_chunks(chunks):
    colors_list = list(COLORS.keys())[2:8]
    print(f"Total Number of Chunks: {len(chunks)}")

    for num, chunk in enumerate(chunks, 1):
        print(f"Chunk {num}: Length {len(chunk)} chars")
        print(colored(text=chunk, color=choice(colors_list)), end="\n\n")

In [43]:
display_chunks([chunk.chunk_text for chunk in chunks])

Total Number of Chunks: 3
Chunk 1: Length 277 chars
Artificial intelligence is transforming technology and shaping the future. Machine learning algorithms are becoming more sophisticated every day. Deep learning models can now process vast amounts of data efficiently. Neural networks are inspired by the human brain's structure.

Chunk 2: Length 283 chars
The best pasta recipes include fresh ingredients and proper cooking techniques. Italian cuisine emphasizes quality olive oil and regional cheeses. Authentic carbonara uses guanciale, eggs, pecorino romano, and black pepper. Cooking pasta al dente ensures the best texture and flavor.

Chunk 3: Length 260 chars
Climate change is affecting ecosystems worldwide. Rising temperatures are causing glaciers to melt at unprecedented rates. Scientists warn that immediate action is needed to reduce carbon emissions. Renewable energy sources offer hope for a sustainable future.



In [44]:
response.chunks[2]

Chunk(chunk_text='Climate change is affecting ecosystems worldwide. Rising temperatures are causing glaciers to melt at unprecedented rates. Scientists warn that immediate action is needed to reduce carbon emissions. Renewable energy sources offer hope for a sustainable future.', summary='The impact of global climate change and the urgent need for carbon reduction and renewable energy solutions.')

In [45]:
# create documents from chunks

docs = [Document(page_content=chunk.chunk_text, metadata={"summary": chunk.summary}) for chunk in chunks]

In [49]:
print(docs)

[Document(metadata={'summary': 'An overview of how artificial intelligence, machine learning, deep learning, and neural networks are advancing and transforming technology.'}, page_content="Artificial intelligence is transforming technology and shaping the future. Machine learning algorithms are becoming more sophisticated every day. Deep learning models can now process vast amounts of data efficiently. Neural networks are inspired by the human brain's structure."), Document(metadata={'summary': 'Key elements of Italian cuisine and essential tips for cooking authentic, high-quality pasta dishes.'}, page_content='The best pasta recipes include fresh ingredients and proper cooking techniques. Italian cuisine emphasizes quality olive oil and regional cheeses. Authentic carbonara uses guanciale, eggs, pecorino romano, and black pepper. Cooking pasta al dente ensures the best texture and flavor.'), Document(metadata={'summary': 'The impact of global climate change and the urgent need for car

In [48]:
print(docs[1])

page_content='The best pasta recipes include fresh ingredients and proper cooking techniques. Italian cuisine emphasizes quality olive oil and regional cheeses. Authentic carbonara uses guanciale, eggs, pecorino romano, and black pepper. Cooking pasta al dente ensures the best texture and flavor.' metadata={'summary': 'Key elements of Italian cuisine and essential tips for cooking authentic, high-quality pasta dishes.'}
